# P05 Làm sạch dữ liệu đơn giản

Bản thay thế theo yêu cầu chỉ clean data. P06 bỏ. Mỗi bước có comment trong code và lời giải thích ngay sau bước đó. Đọc dữ liệu gốc, sửa đúng 14 ô trống, giữ số dòng và cột gốc. Dùng từ **tập train**.

**Cách chạy:** mở notebook trong thư mục CleanOnly, chọn Restart Kernel and Run All. Các file được xuất vào output. Có thể làm cùng thao tác bằng workflow P05_Clean_Data.ows.


## 1 Đọc ba file gốc

In [ ]:
from pathlib import Path
import pandas as pd
root = Path.cwd()
train = pd.read_csv(root / "input/train.csv", dtype={"StateHoliday": "string"})
test = pd.read_csv(root / "input/test.csv", dtype={"StateHoliday": "string"})
store = pd.read_csv(root / "input/store.csv")
# Đọc riêng từng bảng để giữ nguyên cấu trúc gốc, không ghép hoặc tạo cột mới.
print("Train, test, store:", train.shape, test.shape, store.shape)

Lý do: train là lịch sử bán hàng, test là các ngày cần dự báo, store là thông tin cửa hàng. Giữ riêng 3 bảng để người làm bước sau tự chọn cách ghép. Đọc StateHoliday dưới dạng chữ để mã 0 không bị tách thành hai loại. Mở notebook từ thư mục CleanOnly để đường dẫn input hoạt động.

## 2 Kiểm tra ô trống và dòng trùng

In [ ]:
for name, data in [("train", train), ("test", test), ("store", store)]:
    print(name, "missing:", data.isna().sum().loc[lambda x: x > 0].to_dict())
    key = ["Store"] if name == "store" else ["Store", "Date"]
    assert not data.duplicated(key).any(), f"{name}: cần kiểm tra khóa bị trùng"
# Không có khóa trùng nên không xóa dòng. Nếu nguồn thay đổi, dừng để kiểm tra.

Lý do: cần biết chính xác ô nào bị thiếu trước khi điền. Cùng Store và Date chỉ được có một dòng. Kết quả hiện tại không có khóa trùng, nên không xóa dòng nào. Train không thiếu ô; test thiếu 11 Open; store thiếu 3 khoảng cách và một số thông tin lịch chưa biết/không áp dụng.

## 3 Kiểm tra lịch mở cửa rồi điền Open

In [ ]:
missing = test["Open"].isna()
history = train.loc[(train.Store == 622) & (train.DayOfWeek <= 6) & (train.StateHoliday == "0"), "Open"]
assert missing.sum() == 11 and test.loc[missing, "Store"].eq(622).all()
assert test.loc[missing, "DayOfWeek"].le(6).all() and test.loc[missing, "StateHoliday"].eq("0").all()
assert len(history) > 0 and history.eq(1).all()
print("Ngày lịch sử phù hợp:", len(history), "Tỷ lệ mở:", history.mean())
test.loc[missing, "Open"] = 1
# Chỉ điền 11 ô thiếu vì cửa hàng 622 luôn mở vào ngày thường tương ứng trong train.

Lý do: 11 ô trống đều thuộc Store 622 vào thứ Hai đến thứ Bảy, không phải ngày lễ. Trong tập train có 784 ngày cùng điều kiện và tất cả đều mở. Vì vậy chọn Open = 1 cho các ô trống. Đây là ước lượng theo lịch sử, không phải xác nhận cửa hàng thực sự mở trong tương lai. Không sửa bất kỳ Open nào đã có giá trị. Nếu bộ dữ liệu khác, các lệnh assert sẽ yêu cầu kiểm tra lại thay vì tự áp dụng quy tắc này.

Đối chiếu EDA mục 1.12: EDA tính cả ngày lễ nên tỷ lệ mở cửa thứ Hai đến thứ Bảy khoảng 94,07%–100%. Ở đây chỉ xét ngày không lễ giống 11 dòng test bị thiếu, nên 784/784 ngày mở = 100%. Cùng kết luận điền Open = 1; không có mâu thuẫn về dữ liệu.

## 4 Điền khoảng cách bằng trung vị

In [ ]:
distance = store["CompetitionDistance"].median()
store["CompetitionDistance"] = store["CompetitionDistance"].fillna(distance)
# Trung vị ít bị kéo lên bởi vài khoảng cách rất lớn; không hiểu ô thiếu là 0 mét.
print("Khoảng cách dùng để điền:", distance, "mét")

Lý do: lấy giá trị đứng giữa các khoảng cách đã biết trong store, bằng 2.325 mét. Điền cùng giá trị này cho 3 ô thiếu ở Store 291, 622 và 879. Đây là cách đơn giản, dễ giải thích và có thể đặt Fixed value = 2325 trong Orange. Khác bản trước: không chia theo StoreType nữa. Khoảng cách đã có được giữ nguyên. Không thêm cờ, không log và không chuẩn hóa.

EDA mục 1.1 chỉ xác định 3 khoảng cách bị thiếu, chưa quy định giá trị điền. Trung vị 2.325 mét là lựa chọn của bước P05 để đơn giản hóa, không phải con số được trích từ EDA.

## 5 Giữ các ô không áp dụng hoặc chưa biết

In [ ]:
assert store.loc[store.Promo2 == 0, ["Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"]].isna().all().all()
# Không tham gia Promo2 thì không có lịch tham gia; tháng/năm đối thủ chưa biết cũng giữ trống.
assert test.Open.notna().all() and store.CompetitionDistance.notna().all()

Lý do: 544 cửa hàng không tham gia Promo2 nên không có tuần, năm và lịch Promo2. 354 cửa hàng chưa biết tháng/năm mở cửa đối thủ. Không tự tạo lịch giả bằng cách điền 0 hoặc trung bình. Dataset sạch vẫn có thể có những ô trống được giải thích rõ.

## 6 Xuất ba file sạch

In [ ]:
(root / "output").mkdir(exist_ok=True)
for name, data in [("train", train), ("test", test), ("store", store)]:
    data.to_csv(root / "output" / f"{name}_clean.csv", index=False, encoding="utf-8-sig")
    print(f"{name}_clean.csv:", data.shape)
# Giữ toàn bộ dòng và cột gốc; P06 bỏ, không sampling, không Feature Engineering.

Lý do: lưu output riêng để không đè file gốc. Kết quả gồm train_clean.csv (1.017.209 dòng, 9 cột), test_clean.csv (41.088 dòng, 8 cột), store_clean.csv (1.115 dòng, 10 cột). Customers, Sales và các trường lịch vẫn còn nguyên. Không thêm biến, không ghép bảng, không chọn biến mô hình và không tạo mẫu.

## Các bước không làm trong bản này

- P06: bỏ sampling, dùng đầy đủ dữ liệu.
- P07–P09: kiểm tra khóa và cách đọc mã ngày lễ; không ghép store vào train/test.
- P10–P11: chưa chọn/xóa biến cho mô hình; Customers vẫn ở train_clean. Khi forecast sau này phải loại Customers khỏi predictor vì test không có cột này.
- P12–P13: không tự sửa doanh số lạ, 54 ngày mở cửa có Sales = 0 hoặc năm đối thủ 1900/1961; chưa có bằng chứng là lỗi.
- P14: không log/scale vì bản này chỉ làm sạch, chưa chuẩn bị biến cho thuật toán.

Không dùng các file P14 có 22 cột của bản trước cho lần bàn giao này. Các ô đã điền được ghi ngoài dataset trong checks/changed_cells.csv. Không có cột cờ mới trong ba file sạch.


## Đối chiếu với EDA ngày 20/10/2026

Đã kiểm tra **56/56 đối chiếu số liệu được chọn** với U02_EDA_RootCause.ipynb. Ba CSV sạch và code 33 dòng không thay đổi trong đợt kiểm tra. Chi tiết nằm ở [EDA_MATCH.md](EDA_MATCH.md).

- 180 cửa hàng thiếu nửa cuối năm 2014; Store 988 thiếu riêng ngày 01/01/2013. Phần giải thích “181 cửa hàng bắt đầu muộn năm 2013” trong EDA cần sửa.
- EDA đếm 191 đối thủ mở đến 30/09/2015; nếu chỉ tính đến cuối train 31/07/2015 thì là 188.
- Mốc Promo2 trong train là 212 nếu đổi tuần ISO đúng, khác 225 theo cách đổi tuần trong EDA. Không tạo hoặc sửa cột tuần/năm trong bản clean.
- Ví dụ Store 6 ở EDA mục 3.3 trả về bảng rỗng vì Promo2 = 0, nên chưa phải kiểm thử thành công.

Các bộ lọc để tính trung bình trên ngày mở, xử lý riêng 54 Sales = 0 và log Sales thuộc phần phân tích/mô hình. File clean nền vẫn giữ đủ dữ liệu theo yêu cầu. Không gọi các bước đó là đã thực hiện trong P05.
